https://medium.com/thedeephub/building-clip-model-from-scratch-using-pytorch-contrastive-learning-image-pretraining-4cac7c298586

https://github.com/moein-shariatnia/OpenAI-CLIP

https://github.com/mishra-18/ML-Models/blob/main/clip.py

https://medium.com/correll-lab/building-clip-from-scratch-68f6e42d35f4

## CLIP 的架構支援零樣本學習，通過利用圖像和文本之間廣泛的學習關聯來執行未直接訓練的任務。

例如，根據它們的文本描述，它可以對訓練期間從未見過的圖片進行分類。

論文中提到 ：「我們在 ImageNet 零樣本上匹配了原始 ResNet-50 的準確性，而無需使用它所訓練的 128 萬個訓練樣本中的任何一個。

> In 2021 OpenAI's paper: “Learning Transferable Visual Models From Natural Language Supervision"

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision
from transformers import DistilBertTokenizer, DistilBertModel, DistilBertConfig
from PIL import Image
import numpy as np
import torch.nn.functional as F

In [ ]:
class TextEncoder(nn.Module):
    def __init__(self, embed_dim, proj_dim):
        super().__init__()
        self.model = DistilBertModel(config=DistilBertConfig())
        self.projection = nn.Linear(embed_dim, proj_dim)
        self.layer_norm = nn.LayerNorm(proj_dim)

    def forward(self, input_ids, attention_mask):
        x = self.model(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        x = x[:, 0, :] # B, T[cls], E
        x = self.projection(x)
        return self.layer_norm(x)

In [ ]:
class ImageEncoder(nn.Module):
    def __init__(self, base_model, embed_dim, proj_dim):
        super().__init__()

        self.model = base_model

        for param in self.model.parameters():
            param.requires_grad = True

        self.projection = nn.Linear(embed_dim, proj_dim)
        self.layer_norm = nn.LayerNorm(proj_dim)

    def forward(self, x):
        x = self.projection(self.model(x))
        return self.layer_norm(x)

In [ ]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
texts = ["This is a sample sentence.", "This is another example."]
inputs= tokenizer(texts, padding=True, truncation=True, return_tensors="pt").to(device) 

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, texts, image_paths):

        self.image_paths = image_paths
        self.texts = texts
        tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
        self.inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt") 
        self.transform = torchvision.transforms.ToTensor()
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path)
        image = self.transform(image)

        caption, mask = self.inputs[idx].items()
        
        return {
            "image": image,
            "input_ids": caption["input_ids"],
            "mask": mask["attention_mask"]
        }

In [ ]:
encoder = ImageEncoder(embed_dim=768, proj_dim=256)
inputs = encoder(inputs['input_ids'], inputs['mask'])

In [ ]:
class CLIPModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        ViT = VissionTransformer( 
            num_layers=8,
            img_size=224,
            emb_size=768,
            patch_size=16,
            num_head=6,
            num_class=768).to(self.device)
        self.image_encoder = ImageEncoder(base_model=ViT, embed_dim=768, proj_dim=256)
        self.text_encoder = TextEncoder(embed_dim=768, proj_dim=256)